## Setup notes

- **Validation data.** `data/test.csv` is the Kaggle test file: it has only `id` and `text`, no labels, so no AUROC can be computed on it (the assignment also says to use *labeled validation data*). Validation here is therefore a **stratified 20% hold-out of `data/train.csv`** (`random_state=42`), the same split the team used in Part 4 (`fangfeidong/part4.py`). The model is trained on the remaining 8,000 rows and never sees the 2,000 validation rows.
- **Features.** 768-d embeddings from `embeddinggemma-2:270m` served by a local Ollama daemon (`ollama serve`), batched as in `lr.ipynb`. Embeddings are cached to `data/train_embeddinggemma2_270m.npy` so re-runs skip the ~6 min embedding step.
- **Model.** `SVC(kernel="poly", C=10.0)`. `probability=True` is added only to obtain P(AI) (Platt scaling) for the error analysis; it does not change the fitted decision boundary.

In [1]:
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import ollama

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# sklearn 1.9 deprecates SVC(probability=True); it still works and matches the team's setup
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")

RANDOM_SEED = 42
EMBED_MODEL = "embeddinggemma-2:270m"
THRESHOLD = 0.5  # P(AI) >= 0.5 -> predicted AI

df = pd.read_csv("./data/train.csv")
print(df.shape, df["label"].value_counts().to_dict())
print("test.csv columns:", list(pd.read_csv("./data/test.csv", nrows=1).columns), "(no label column)")

(10000, 3) {0: 5000, 1: 5000}
test.csv columns: ['id', 'text'] (no label column)


## Embed with EmbeddingGemma2:270M

In [2]:
def embed_texts(texts, model=EMBED_MODEL, batch_size=32):
    """Embed texts with Ollama in small batches (same approach as lr.ipynb)."""
    texts = np.asarray(texts, dtype=str).tolist()
    vectors = []
    for start in range(0, len(texts), batch_size):
        response = ollama.embed(model=model, input=texts[start : start + batch_size])
        vectors.extend(response["embeddings"])
    return np.asarray(vectors, dtype=np.float32)


CACHE_PATH = "./data/train_embeddinggemma2_270m.npy"

if os.path.exists(CACHE_PATH):
    X = np.load(CACHE_PATH)
    print(f"Loaded cached embeddings from {CACHE_PATH}")
else:
    start_time = time.time()
    X = embed_texts(df["text"])
    np.save(CACHE_PATH, X)
    print(f"Embedding time: {time.time() - start_time:.1f} seconds")

assert len(X) == len(df)
print(X.shape)

Loaded cached embeddings from ./data/train_embeddinggemma2_270m.npy
(10000, 768)


## Train / validation split and model

In [3]:
y = df["label"].to_numpy()
indices = np.arange(len(df))

train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
print("train:", X_train.shape, "val:", X_val.shape)

model = SVC(kernel="poly", C=10.0, probability=True, random_state=RANDOM_SEED)

start_time = time.time()
model.fit(X_train, y_train)
print(f"Training time: {time.time() - start_time:.1f} seconds")

train: (8000, 768) val: (2000, 768)
Training time: 33.2 seconds


## Q4.1 — training vs validation AUROC

In [4]:
p_train = model.predict_proba(X_train)[:, 1]
p_val = model.predict_proba(X_val)[:, 1]

train_auroc = roc_auc_score(y_train, p_train)
val_auroc = roc_auc_score(y_val, p_val)

print(f"Train AUROC: {train_auroc:.4f}   accuracy@{THRESHOLD}: {accuracy_score(y_train, p_train >= THRESHOLD):.4f}")
print(f"Val   AUROC: {val_auroc:.4f}   accuracy@{THRESHOLD}: {accuracy_score(y_val, p_val >= THRESHOLD):.4f}")
print(f"AUROC gap (train - val): {train_auroc - val_auroc:.4f}")

# evidence for the gap discussion: how much of the training set the SVM memorises
n_sv = int(model.n_support_.sum())
print(f"Support vectors: {n_sv} of {len(X_train)} training rows ({n_sv / len(X_train):.1%})")
print(f"Features: {X_train.shape[1]} dims, default degree-{model.degree} polynomial kernel")

Train AUROC: 1.0000   accuracy@0.5: 1.0000
Val   AUROC: 0.9722   accuracy@0.5: 0.9200
AUROC gap (train - val): 0.0278
Support vectors: 3875 of 8000 training rows (48.4%)
Features: 768 dims, default degree-3 polynomial kernel


## Q4.2 — validation AUROC by text length

Bins are defined on the whitespace-separated word count of each passage: **short** (< 160 words), **medium** (160–219), **long** (≥ 220). The cut points are roughly the 25th and 55th percentiles, so every bin has ≥ 500 rows and both classes. 95% intervals are from 1,000 bootstrap resamples within each bin.

In [5]:
val = df.iloc[val_idx].copy()
val["p_ai"] = p_val
val["pred"] = (val["p_ai"] >= THRESHOLD).astype(int)
val["n_words"] = val["text"].str.split().str.len()

print(val["n_words"].describe().round(1).to_string())

BINS = [0, 160, 220, np.inf]
BIN_LABELS = ["short (<160 words)", "medium (160-219 words)", "long (>=220 words)"]
val["length_bin"] = pd.cut(val["n_words"], bins=BINS, labels=BIN_LABELS, right=False)


def bootstrap_auroc_ci(labels, scores, n_boot=1000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    labels, scores = np.asarray(labels), np.asarray(scores)
    stats = []
    for _ in range(n_boot):
        sample = rng.integers(0, len(labels), len(labels))
        if labels[sample].min() == labels[sample].max():
            continue
        stats.append(roc_auc_score(labels[sample], scores[sample]))
    return np.percentile(stats, [2.5, 97.5])


rows = []
for name, group in val.groupby("length_bin", observed=True):
    low, high = bootstrap_auroc_ci(group["label"], group["p_ai"])
    rows.append({
        "bin": name,
        "count": len(group),
        "AI share": round(group["label"].mean(), 3),
        "val AUROC": round(roc_auc_score(group["label"], group["p_ai"]), 4),
        "95% CI": f"[{low:.3f}, {high:.3f}]",
        "accuracy": round(accuracy_score(group["label"], group["pred"]), 3),
        "FP": int(((group["pred"] == 1) & (group["label"] == 0)).sum()),
        "FN": int(((group["pred"] == 0) & (group["label"] == 1)).sum()),
    })
length_table = pd.DataFrame(rows)
length_table

count    2000.0
mean      207.4
std        62.9
min        76.0
25%       159.0
50%       205.0
75%       254.0
max       350.0


,bin,count,AI share,val AUROC,95% CI,accuracy,FP,FN
0,short (<160 words),508,0.579,0.9491,"[0.931, 0.965]",0.894,20,34
1,medium (160-219 words),635,0.400,0.9739,"[0.961, 0.985]",0.915,38,16
2,long (>=220 words),857,0.527,0.9847,"[0.977, 0.990]",0.939,16,36


> Since the embedding model project paragaphs of different length to fixed size vector (768 dim). Shorter paragraph might provide less clue to be encoded, leading to less meaningful feature extracted and thus poorer performance on the same trained model. Under that circumstance, a few keyword might over-dominantly influence the embedding and thus trick the model. We see that less of a issue as length of of text increase, and the model is provided more detail to distinguish.